# IDS Challenge

## 01 Optimization: Maximum Coverage of QA Rounds by the New Robot Fleet

The introduction of walking robots in the maintenance department was a success.
The company has therefore already purchased **three walking robots**, which are now also meant to support
quality assurance (QA) in collecting material samples.
Each robot can operate for a maximum of **5 hours straight** before its battery needs to be recharged.

Until now, three employees on the morning shift (Gianni, Lissi, and Ben) decided themselves which machines
each of them would visit to collect material samples. In total, so many machines need to be visited that
even three robots with their limited battery runtime **cannot visit all of them within a single shift**.
Management would therefore like to know the following from you:

1. **How many of the machines can the three robots cover in total at most**, if each robot drives at most one
   round trip of max. 5 hours? Which three round trips (sequences) achieve this maximum coverage?
2. Which machines remain uncovered and still need to be visited on foot by the employees? Each remaining
   machine is visited individually from the depot and back (not as one combined round trip covering several
   remaining machines). The **sum of these individual round trips** (depot -> machine -> depot) over all
   remaining machines should be as small as possible. So it's not just *how many* machines the robots take
   over that matters, but also *which* ones - a selection that covers many but poorly located machines can be
   worse for the remaining team than a selection that covers a few less, but machines closer to the depot.

So there are two linked objectives: first maximize the number of machines covered by the robots, and among
the selections/tours that achieve this, try to find the one that keeps the sum of depot distances of the
remaining machines as small as possible. Please also describe how you dealt with this trade-off.
A round trip always starts and ends at the quality assurance lab (also called the depot).

The following data is provided:
* A set of machines with locations from which material samples need to be collected.
* The three sets of machines that were previously visited by the three employees during their early shift.
  (The sequence the employees used is different and unfortunately unknown.)
* A travel-service-time matrix (German: Laufzeit-Servicezeit-Matrix: LSM), which contains the travel times
  between the machines and the service times at each machine (i.e., the time the robotic arm needs at each
  machine to place the sample into the box on the back of the walking robot).

Joshi, from the previous team, has already done some preliminary work to help you solve the task.
These preparations are included in this Jupyter notebook.

Joshi has already implemented a function that calculates the total time of a sequence. You will find the
function `calc_total_tour_time()` further below.

This document contains the preparations and a short message from the previous team!

# Our Preparations

There are already several files available to help you solve the task. They are all located in the ``data/`` subfolder.

### List of Machines for the Three Employees from the QA Team: Gianni, Lissi, and Ben

The files `data/points_gianni_55_42.txt`, `data/points_lissi_55_42.txt`, and `data/points_ben_55_42.txt` contain all the machines that each of the three visited.

The contents of the files look like this:

```
0; 0; 450.0; 0  
43; 25.696035620363634; 437.3914169849524; 0  
44; 293.39534810320026; 479.4355525765641; 1  
...
```

The first number is the index of the machine (or point), the second number is the x-coordinate, the third number is the y-coordinate, and the fourth number is the floor.
The start and end point of each robot tour is the quality assurance lab. This start/end point is called the depot and has index 0 (and appears in the first line of all three files).

### Travel–Service-Time Matrix

To calculate the total tour duration, we need a travel–service-time matrix in addition to the tour sequence.
We have stored the LS matrix in the text file `data/ls_matrix_55_42.txt`.

The file looks like this:

```
0; 0; 0  
0; 1; 1000.0  
0; 2; 1000.0  
0; 3; 1123.606797749979  
...
```

Each line represents a connection (edge) between two points (machines or the depot).
The first number is the index of the start point of the connection, the second number is the index of the end point, and the third number
is the travle time including the service time at the machine (i.e., the end point of the connection), in seconds.
If the endpoint of a connection is the depot, then of course no service time is added.

### Calculating Tour Duration Based on a Sequence

We have already implemented a function that calculates the tour duration for a given sequence.
For this, we created a small example matrix with only 4 machines plus the start/end point.
We represent the distance matrix as a dictionary.
A function that reads the text files has unfortunately not been implemented yet...

So you can use the function `calc_total_tour_time` to calculate the tour duration for a given sequence.

In [1]:
def calc_total_tour_time(sequ, matrix):
    total_time = 0
    for i in range(len(sequ)-1):
        total_time += matrix[sequ[i], sequ[i+1]]
    return total_time

Here is the test of the function with our small distance matrix:

In [5]:
lsm_dict = {(0, 0): 0, (0, 1): 1000.0, (0, 2): 1000.0, (0, 3): 1123.606797749979, (0, 4): 1123.606797749979, (1, 0): 1000.0, (1, 1): 0, (1, 2): 1100.0, (1, 3): 1182.842712474619, (1, 4): 1100.0, (2, 0): 1000.0, (2, 1): 1100.0, (2, 2): 0, (2, 3): 1100.0, (2, 4): 1182.842712474619, (3, 0): 1123.606797749979, (3, 1): 1182.842712474619, (3, 2): 1100.0, (3, 3): 0, (3, 4): 1100.0, (4, 0): 1123.606797749979, (4, 1): 1100.0, (4, 2): 1182.842712474619, (4, 3): 1100.0, (4, 4): 0}

sequence = [0, 2, 3, 4, 1, 0]


tour_time_sec = calc_total_tour_time(sequence, lsm_dict)

print(f"Total time for the tour: {tour_time_sec} seconds")
print(f"Total time for the tour: {tour_time_sec/60} minutes")
print(f"Total time for the tour: {tour_time_sec/3600} hours")

Total time for the tour: 5300.0 seconds
Total time for the tour: 88.33333333333333 minutes
Total time for the tour: 1.4722222222222223 hours


# Our Initial Thoughts

## Getting an Overview

As a first step, we wanted to get an overview and collect all the points from the list of the three team members into a single list.
Since the data consists of x-y coordinates, we wanted to create a 2D scatter plot of the points and color them differently depending on which floor the machine is on.
We also wanted to label the points with their indices so we can later tell which machine is which.
Additionally, we could use different markers for the points depending on whether they belong to Gianni, Lissi, or Ben.
Maybe an LLM can help you with creating the scatter plot.

## Algorithm to Determine the Sequence

We're not very experienced with optimization.
We've heard there are algorithms that calculate the optimal sequence (sequences are often also called tours), i.e., the shortest possible sequence given a travel-service-time matrix.
But we would first implement a simple algorithm that builds a sequence step-by-step by always attaching the nearest not-yet-visited neighbor.
This time, however, we also need to check after every step whether the 5 hours are still enough: before attaching another machine, we check whether the tour so far plus the trip to the next machine plus the trip back to the depot still fits within the 5 hours.
If not, we stop the tour and close it by returning to the depot - that is then the finished tour for one robot.
We could repeat this for all three robots - but then we'd have to make sure that all three robots don't keep starting at the same, nearest machines.
Alternatively, you could try to find established algorithms (but be aware that you should understand them and be able to explain them in the presentation).

We thought it might be a good idea to test things with a very simple setup.
We imagined a small distance matrix with just 3 machines and the start/end point.
We assumed we are in a 2D space without obstacles and the start/end point is at position (0,0) (index 0), and the machines are at (10,10) (index 1), (0,10) (index 2), and (10,0) (index 3).
With that, we could compute a travel time matrix using Euclidean distance (since there are no paths or obstacles), and we could easily draw it.
Let's just assume that one distance unit is traveled in one second.
The shortest sequence would then simply be the sequence `[0, 2, 1, 3, 0]` or in reverse `[0, 3, 1, 2, 0]`.

Simple examples like this often help us understand how we need to implement the algorithms.
If it works for the simple example, then we can think of something slightly more complex and eventually apply it to the real data.

Maybe you'll find a way to plot the solution on the scatter plot.

Just a reminder: the formula for Euclidean distance between two points is
$\sqrt{(x_1 - x_2)^2 + (y_1 - y_2)^2}$

## Splitting Across the Three Robots

We also noticed when observing the quality assurance team members that the machine assignments weren't particularly well-balanced - maybe you'll see that in your scatter plot too.
You could split the machines into three spatially sensible groups (one group per robot) and then determine a separate tour for each group.
Each group probably won't completely fit within a robot's 5 hours - so a few machines will remain in each group.
For the machines that remain across all three groups, in the end only the sum of the individual round trips from the depot counts - how you keep this sum as small as possible is part of the task.

These are our ideas, but maybe you have completely different ideas and find better approaches and algorithms!

Good luck!
Your Joshi!

### Acceptance Criteria

* Jupyter notebook `submission_tp01_optimization.ipynb` including a description of the basic idea and the implementation of one or more algorithms
* Use of functions to write clean and readable code (Tip: a function should not exceed 10 lines of code. If it does, it should be broken down into smaller functions)
* Estimation of the runtime complexity of the algorithms (n = number of machines)
* Three round trips for the robots, each respecting the 5-hour limit, plus a statement of how many machines are covered in total
* List of the remaining machines, together with the sum of the individual round trips from the depot to these machines, kept as small as possible
* A clear description of how you handled the trade-off between maximum coverage and a small distance sum for the remaining machines
* Clear and appealing presentation of the algorithms and results during the presentation (10 minutes total)